In [ ]:
%pip install torch xgboost transformers textblob -q
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import xgboost as xgb
import yfinance as yf
from transformers import BertTokenizer, BertForSequenceClassification
from textblob import TextBlob
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, accuracy_score
import matplotlib.pyplot as plt

# 1. Configuration & Tickers
TICKERS = ['0700.HK', '0005.HK', '1299.HK', '3690.HK'] # Tencent, HSBC, AIA, Meituan [1]
START_DATE = "2022-01-01"
END_DATE = "2025-12-31"
WINDOW_SIZE = 10
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Polyvalent Sentiment Extraction Engine
# Implements the multidimensional affect extraction described in Section 3.2 [1]
class PolyvalentExtractor:
    def __init__(self):
        # Using FinBERT for contextual polarity [2, 3]
        self.tokenizer = BertTokenizer.from_pretrained('ProsusAI/finbert')
        self.model = BertForSequenceClassification.from_pretrained('ProsusAI/finbert').to(DEVICE)
        
    def get_polyvalent_features(self, text):
        # A. Contextual Polarity (FinBERT)
        inputs = self.tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(DEVICE)
        with torch.no_grad():
            outputs = self.model(**inputs)
            probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
            polarity_val = torch.argmax(probs).item() - 1 # Map to -1, 0, 1
            
        # B. Subjectivity (TextBlob) [4, 1]
        blob = TextBlob(text)
        subjectivity = blob.sentiment.subjectivity
        
        # C. Intensity (Threshold-based magnitude) [1]
        # Magnitude based on probabilistic confidence of the sentiment head
        intensity = torch.max(probs).item()
        
        return np.array([polarity_val, subjectivity, intensity])

# 3. Feature Engineering & Technical Indicators
def get_technical_indicators(df):
    # RSI calculation
    delta = df['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / loss
    df = 100 - (100 / (1 + rs))
    
    # MACD [1]
    df['EMA12'] = df['Close'].ewm(span=12, adjust=False).mean()
    df['EMA26'] = df['Close'].ewm(span=26, adjust=False).mean()
    df = df['EMA12'] - df['EMA26']
    
    return df.fillna(method='bfill')

# 4. Modeling Architectures

# A. LSTM Architecture [2, 5]
class StockLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, num_layers=2):
        super(StockLSTM, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(hidden_dim, 1)
        
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

# B. Transformer Architecture (Peak performer: 64.7% accuracy)
class StockTransformer(nn.Module):
    def __init__(self, input_dim, nhead=4, num_layers=2, dim_feedforward=256):
        super(StockTransformer, self).__init__()
        self.embedding = nn.Linear(input_dim, dim_feedforward)
        encoder_layer = nn.TransformerEncoderLayer(d_model=dim_feedforward, nhead=nhead, dropout=0.1)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Linear(dim_feedforward, 1)

    def forward(self, x):
        x = self.embedding(x).transpose(0, 1) # Seq-len first
        out = self.transformer_encoder(x)
        return self.fc(out[-1, :, :])

# 5. Pipeline Execution Logic
def run_forecasting_pipeline(ticker):
    # Data Retrieval
    data = yf.download(ticker, start=START_DATE, end=END_DATE)
    data = get_technical_indicators(data)
    
    # Feature Scaling
    scaler = MinMaxScaler()
    scaled_data = scaler.fit_transform([data])
    
    # Note: In a live environment, sentiment vectors would be merged here [1]
    # Simulated alignment for the 771-dimensional sentiment vector
    
    # Training (2022-2024) vs Testing (2025) [6, 1]
    train_size = int(len(scaled_data) * 0.8)
    train_data = scaled_data[:train_size]
    test_data = scaled_data[train_size:]
    
    print(f"Pipeline executed for {ticker}. Ready for model training.")
    return train_data, test_data

if __name__ == "__main__":
    for ticker in TICKERS:
        run_forecasting_pipeline(ticker)

ModuleNotFoundError: No module named 'torch'